# N-BEATS Final Experiment: Hyperparameter Grid Search

This notebook runs the final N-BEATS experiment after the single-change experiments. It performs a controlled hyperparameter grid search and compares two training objectives:

- regular L1 loss;
- holiday-aware weighted L1 loss.

The model still uses plain N-BEATS with historical `Weekly_Sales` as input. External tabular features are not added as direct model inputs.

## 1. Environment setup

In [ ]:
%pip install -q "torch>=2.3,<3" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped. This is expected outside Colab.')

In [ ]:
import itertools
import math
import os
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import wandb

SEED = 42
WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
WANDB_GROUP = "nbeats-final-grid-search"

DATA_DIR_CANDIDATES = [
    Path('/content/drive/MyDrive/walmart_competition_data'),
    Path('/content/drive/My Drive/walmart_competition_data'),
    Path('/content/walmart_competition_data'),
    Path('../../data'),
    Path('data'),
]

OUTPUT_DIR = Path('/content/artifacts/nbeats_final_grid') if Path('/content').exists() else Path('artifacts/nbeats_final_grid')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'validation_weeks': 32,
    'context_length': 52,
    'forecast_horizon': 32,
    'min_series_length': 84,
    'max_epochs': 100,
    'early_stopping_patience': 8,
    'num_blocks': 4,
    'num_layers': 4,
    'holiday_weight': 5.0,
    'clip_grad_norm': 1.0,
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

HYPERPARAMETER_GRID = {
    'optimizer': ['sgd', 'adam'],
    'loss_type': ['regular_l1', 'holiday_weighted_l1'],
    'batch_size': [64, 128],
    'learning_rate': [1e-3, 3e-4],
    'hidden_units': [128, 256],
    'dropout': [0.0, 0.10],
    'weight_decay': [0.0, 1e-4],
}

# Use a small integer while debugging. Keep None for the full final grid search.
MAX_GRID_RUNS = None

BASELINE_REFERENCE_WMAE = 2157.9829
EXPERIMENT_1_WMAE = 2186.5015
EXPERIMENT_2_WMAE = 2662.8061
EXPERIMENT_3_WMAE = 2185.1366


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)
print(f"Using device: {CONFIG['device']}")
print(f"Python: {platform.python_version()}")
print(f"Torch: {torch.__version__}")

## 2. Load data

In [ ]:
def resolve_data_dir(candidates):
    for candidate in candidates:
        if candidate.exists() and (candidate / 'train.csv').exists():
            return candidate
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(f'Could not find train.csv. Searched:\n{searched}')


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f'Data directory: {DATA_DIR}')

train_raw = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['Date'])
test_raw = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['Date'])
features_raw = pd.read_csv(DATA_DIR / 'features.csv', parse_dates=['Date'])
stores_raw = pd.read_csv(DATA_DIR / 'stores.csv')

required_train_cols = {'Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday'}
missing_cols = required_train_cols.difference(train_raw.columns)
if missing_cols:
    raise ValueError(f'train.csv is missing required columns: {sorted(missing_cols)}')

print('train shape:', train_raw.shape)
print('test shape:', test_raw.shape)
print('features shape:', features_raw.shape)
print('stores shape:', stores_raw.shape)
print('date range:', train_raw['Date'].min().date(), 'to', train_raw['Date'].max().date())

## 3. Baseline preprocessing and holiday weights

The final grid search returns to the best baseline windowing setup:

- `context_length = 52`
- `forecast_horizon = 32`
- last 32 weeks as validation

Holiday information is not added as a model input. It is only used to build target-horizon weights for the loss function when `loss_type = holiday_weighted_l1`.

In [ ]:
train_df = train_raw.copy()
train_df['Store'] = train_df['Store'].astype(int)
train_df['Dept'] = train_df['Dept'].astype(int)
train_df = train_df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

all_dates = pd.date_range(train_df['Date'].min(), train_df['Date'].max(), freq='W-FRI')
sales_pivot = (
    train_df
    .pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales', aggfunc='sum')
    .reindex(columns=all_dates)
    .sort_index()
)

holiday_by_date = (
    train_df.groupby('Date')['IsHoliday']
    .max()
    .reindex(all_dates)
    .fillna(False)
    .astype(bool)
)

context_length = CONFIG['context_length']
forecast_horizon = CONFIG['forecast_horizon']
validation_weeks = CONFIG['validation_weeks']
min_required_weeks = context_length + forecast_horizon

valid_series_mask = sales_pivot.notna().sum(axis=1) >= min_required_weeks
sales_pivot = sales_pivot.loc[valid_series_mask]
filled_sales = sales_pivot.ffill(axis=1).bfill(axis=1).fillna(0.0)

series_index = filled_sales.index
series_keys = pd.DataFrame(series_index.tolist(), columns=['Store', 'Dept'])

if validation_weeks != forecast_horizon:
    raise ValueError('This notebook expects validation_weeks to equal forecast_horizon.')

validation_start = len(all_dates) - validation_weeks
if validation_start < context_length:
    raise ValueError('Not enough historical weeks before validation. Reduce context_length or validation_weeks.')

train_dates = all_dates[:validation_start]
validation_dates = all_dates[validation_start:]

sales_values_original = filled_sales.to_numpy(dtype=np.float32)
log_sales_values = np.log1p(np.clip(sales_values_original, a_min=0.0, a_max=None)).astype(np.float32)

series_center = log_sales_values[:, :validation_start].mean(axis=1, keepdims=True)
series_scale = log_sales_values[:, :validation_start].std(axis=1, keepdims=True)
series_scale = np.where(series_scale < 1e-6, 1.0, series_scale).astype(np.float32)
normalized_values = ((log_sales_values - series_center) / series_scale).astype(np.float32)

print('weekly dates:', len(all_dates))
print('usable Store-Dept series:', len(series_index))
print('training date range:', train_dates.min().date(), 'to', train_dates.max().date())
print('validation date range:', validation_dates.min().date(), 'to', validation_dates.max().date())
print('sales tensor shape:', normalized_values.shape)

In [ ]:
def make_training_windows(
    values: np.ndarray,
    horizon_weights: np.ndarray,
    context: int,
    horizon: int,
    validation_start_idx: int,
):
    X_windows = []
    y_windows = []
    weight_windows = []
    last_window_start = validation_start_idx - context - horizon
    if last_window_start < 0:
        raise ValueError('Not enough history to create training windows before validation.')

    for series_id in range(values.shape[0]):
        for start in range(last_window_start + 1):
            split = start + context
            target_slice = slice(split, split + horizon)
            X_windows.append(values[series_id, start:split])
            y_windows.append(values[series_id, target_slice])
            weight_windows.append(horizon_weights[target_slice])

    return (
        np.asarray(X_windows, dtype=np.float32),
        np.asarray(y_windows, dtype=np.float32),
        np.asarray(weight_windows, dtype=np.float32),
    )


all_horizon_weights = np.where(
    holiday_by_date.to_numpy(dtype=bool),
    CONFIG['holiday_weight'],
    1.0,
).astype(np.float32)

X_train, y_train, train_target_weights = make_training_windows(
    normalized_values,
    all_horizon_weights,
    context_length,
    forecast_horizon,
    validation_start,
)

X_val = normalized_values[:, validation_start - context_length:validation_start].astype(np.float32)
y_val_original = sales_values_original[:, validation_start:validation_start + forecast_horizon].astype(np.float32)
val_holiday_flags = holiday_by_date.loc[validation_dates].to_numpy(dtype=bool)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('train_target_weights:', train_target_weights.shape)
print('X_val:', X_val.shape)
print('y_val:', y_val_original.shape)
print('validation holiday weeks:', int(val_holiday_flags.sum()))

## 4. Log preprocessing to W&B

In [ ]:
wandb.login(key=os.environ.get('WANDB_API_KEY'), relogin=False)

preprocessing_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    job_type='final_grid_preprocessing',
    name='NBEATS_Final_Grid_Preprocessing',
    config={
        **CONFIG,
        'seed': SEED,
        'model_family': 'N-BEATS',
        'experiment_type': 'final_hyperparameter_grid_search',
        'data_dir': str(DATA_DIR),
        'train_rows': int(len(train_raw)),
        'test_rows': int(len(test_raw)),
        'usable_series': int(len(series_index)),
        'train_windows': int(len(X_train)),
        'total_weeks': int(len(all_dates)),
        'train_start_date': str(train_dates.min().date()),
        'train_end_date': str(train_dates.max().date()),
        'validation_start_date': str(validation_dates.min().date()),
        'validation_end_date': str(validation_dates.max().date()),
        'target_transform': 'clip_negative_then_log1p',
        'series_scaling': 'per_series_standardization_fit_on_training_period',
        'grid': HYPERPARAMETER_GRID,
    },
)
preprocessing_run.log({
    'final_grid/usable_series': len(series_index),
    'final_grid/train_windows': len(X_train),
})
preprocessing_run.finish()

## 5. Model, dataset, and metric definitions

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray | None = None, weights: np.ndarray | None = None):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = None if y is None else torch.from_numpy(y.astype(np.float32))
        self.weights = None if weights is None else torch.from_numpy(weights.astype(np.float32))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        if self.weights is None:
            return self.X[idx], self.y[idx]
        return self.X[idx], self.y[idx], self.weights[idx]


class NBeatsBlock(nn.Module):
    def __init__(self, input_size: int, horizon: int, hidden_units: int, num_layers: int, dropout: float):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_units))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            current_size = hidden_units
        self.net = nn.Sequential(*layers)
        self.backcast_head = nn.Linear(hidden_units, input_size)
        self.forecast_head = nn.Linear(hidden_units, horizon)

    def forward(self, x):
        hidden = self.net(x)
        return self.backcast_head(hidden), self.forecast_head(hidden)


class NBeats(nn.Module):
    def __init__(self, input_size: int, horizon: int, hidden_units: int, num_blocks: int, num_layers: int, dropout: float):
        super().__init__()
        self.horizon = horizon
        self.blocks = nn.ModuleList([
            NBeatsBlock(input_size, horizon, hidden_units, num_layers, dropout)
            for _ in range(num_blocks)
        ])

    def forward(self, x):
        residual = x
        forecast = torch.zeros(x.size(0), self.horizon, device=x.device)
        for block in self.blocks:
            backcast, block_forecast = block(residual)
            residual = residual - backcast
            forecast = forecast + block_forecast
        return forecast


def inverse_transform_predictions(pred_normalized: np.ndarray, center: np.ndarray, scale: np.ndarray) -> np.ndarray:
    pred_log = pred_normalized * scale + center
    pred_sales = np.expm1(pred_log)
    return np.clip(pred_sales, a_min=0.0, a_max=None)


def weighted_mae(y_true: np.ndarray, y_pred: np.ndarray, holiday_flags: np.ndarray, holiday_weight: float = 5.0) -> float:
    weights = np.where(holiday_flags.reshape(1, -1), holiday_weight, 1.0)
    weights = np.broadcast_to(weights, y_true.shape)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def regular_l1_loss(prediction: torch.Tensor, target: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(prediction - target))


def holiday_weighted_l1_loss(prediction: torch.Tensor, target: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    return (torch.abs(prediction - target) * weights).sum() / weights.sum()

## 6. Seasonal naive reference

In [ ]:
seasonal_naive_pred = sales_values_original[:, validation_start - forecast_horizon:validation_start]
baseline_wmae = weighted_mae(y_val_original, seasonal_naive_pred, val_holiday_flags, CONFIG['holiday_weight'])
baseline_mae = mae(y_val_original, seasonal_naive_pred)
baseline_rmse = rmse(y_val_original, seasonal_naive_pred)

print(f'Seasonal naive validation WMAE: {baseline_wmae:.4f}')
print(f'Seasonal naive validation MAE: {baseline_mae:.4f}')
print(f'Seasonal naive validation RMSE: {baseline_rmse:.4f}')

## 7. Final hyperparameter grid search

In [ ]:
def evaluate_model(model: nn.Module, X: np.ndarray, batch_size: int = 512):
    model.eval()
    predictions = []
    loader = DataLoader(WindowDataset(X), batch_size=batch_size, shuffle=False, num_workers=0)
    with torch.no_grad():
        for batch_X in loader:
            batch_X = batch_X.to(CONFIG['device'])
            predictions.append(model(batch_X).cpu().numpy())
    return np.concatenate(predictions, axis=0)


def build_grid(grid: dict, max_runs: int | None = None):
    keys = list(grid.keys())
    values = [grid[key] for key in keys]
    configs = [dict(zip(keys, combo)) for combo in itertools.product(*values)]
    if max_runs is not None:
        configs = configs[:max_runs]
    return configs


def train_one_trial(trial_id: int, total_trials: int, trial_params: dict):
    seed_everything(SEED + trial_id)
    trial_config = {**CONFIG, **trial_params}

    train_loader = DataLoader(
        WindowDataset(X_train, y_train, train_target_weights),
        batch_size=trial_config['batch_size'],
        shuffle=True,
        num_workers=trial_config['num_workers'],
        pin_memory=trial_config['device'] == 'cuda',
    )

    model = NBeats(
        input_size=context_length,
        horizon=forecast_horizon,
        hidden_units=trial_config['hidden_units'],
        num_blocks=trial_config['num_blocks'],
        num_layers=trial_config['num_layers'],
        dropout=trial_config['dropout'],
    ).to(trial_config['device'])

    if trial_config['optimizer'] == 'adam':
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=trial_config['learning_rate'],
            weight_decay=trial_config['weight_decay'],
        )
    elif trial_config['optimizer'] == 'sgd':
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=trial_config['learning_rate'],
            weight_decay=trial_config['weight_decay'],
            momentum=0.9,
        )
    else:
        raise ValueError(f"Unsupported optimizer: {trial_config['optimizer']}")

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    loss_fn = holiday_weighted_l1_loss if trial_config['loss_type'] == 'holiday_weighted_l1' else regular_l1_loss

    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type='final_grid_trial',
        name=f"NBEATS_Final_Grid_Trial_{trial_id:03d}",
        config={
            **trial_config,
            'seed': SEED + trial_id,
            'trial_id': trial_id,
            'total_trials': total_trials,
            'model_family': 'N-BEATS',
            'experiment_type': 'final_hyperparameter_grid_search',
            'baseline_reference_weighted_mae': BASELINE_REFERENCE_WMAE,
            'experiment_1_reference_weighted_mae': EXPERIMENT_1_WMAE,
            'experiment_2_reference_weighted_mae': EXPERIMENT_2_WMAE,
            'experiment_3_reference_weighted_mae': EXPERIMENT_3_WMAE,
            'train_windows': int(len(X_train)),
            'validation_series': int(len(X_val)),
        },
    )
    wandb.watch(model, log='gradients', log_freq=100)

    best_wmae = math.inf
    best_epoch = -1
    best_val_pred = None
    best_state_dict = None
    epochs_without_improvement = 0
    early_stopped = False

    for epoch in range(1, trial_config['max_epochs'] + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y, batch_weights in train_loader:
            batch_X = batch_X.to(trial_config['device'])
            batch_y = batch_y.to(trial_config['device'])
            batch_weights = batch_weights.to(trial_config['device'])

            optimizer.zero_grad(set_to_none=True)
            batch_pred = model(batch_X)
            loss = loss_fn(batch_pred, batch_y, batch_weights)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), trial_config['clip_grad_norm'])
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_pred_normalized = evaluate_model(model, X_val)
        val_pred_original = inverse_transform_predictions(val_pred_normalized, series_center, series_scale)

        validation_wmae = weighted_mae(y_val_original, val_pred_original, val_holiday_flags, trial_config['holiday_weight'])
        validation_mae = mae(y_val_original, val_pred_original)
        validation_rmse = rmse(y_val_original, val_pred_original)
        scheduler.step(validation_wmae)

        run.log({
            'epoch': epoch,
            'train/loss': train_loss,
            'validation/weighted_mae': validation_wmae,
            'validation/mae': validation_mae,
            'validation/rmse': validation_rmse,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })

        if validation_wmae < best_wmae:
            best_wmae = validation_wmae
            best_epoch = epoch
            best_val_pred = val_pred_original.copy()
            best_state_dict = {key: value.detach().cpu() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Trial {trial_id + 1:03d}/{total_trials:03d} | epoch {epoch:03d}/{trial_config['max_epochs']:03d} | "
            f"optimizer={trial_config['optimizer']} loss={trial_config['loss_type']} lr={trial_config['learning_rate']} batch={trial_config['batch_size']} | "
            f"train={train_loss:.5f} val WMAE={validation_wmae:.4f} | "
            f"no improvement={epochs_without_improvement}/{trial_config['early_stopping_patience']}"
        )

        if epochs_without_improvement >= trial_config['early_stopping_patience']:
            early_stopped = True
            print(f"Early stopping trial {trial_id} at epoch {epoch}. Best epoch was {best_epoch}.")
            break

    run.summary['best_epoch'] = best_epoch
    run.summary['best_validation_weighted_mae'] = best_wmae
    run.summary['early_stopped'] = early_stopped
    run.finish()

    return {
        'trial_id': trial_id,
        'best_epoch': best_epoch,
        'best_validation_weighted_mae': best_wmae,
        'early_stopped': early_stopped,
        'val_predictions': best_val_pred,
        'state_dict': best_state_dict,
        **trial_params,
    }


grid_configs = build_grid(HYPERPARAMETER_GRID, MAX_GRID_RUNS)
print(f'Final grid search trials: {len(grid_configs)}')
print(f'Max epochs per trial: {CONFIG["max_epochs"]}')
print(f'Early stopping patience: {CONFIG["early_stopping_patience"]}')

grid_results = []
best_trial = None

for trial_id, trial_params in enumerate(grid_configs):
    result = train_one_trial(trial_id, len(grid_configs), trial_params)
    result_for_table = {key: value for key, value in result.items() if key not in {'val_predictions', 'state_dict'}}
    grid_results.append(result_for_table)

    if best_trial is None or result['best_validation_weighted_mae'] < best_trial['best_validation_weighted_mae']:
        best_trial = result


grid_results_df = pd.DataFrame(grid_results).sort_values('best_validation_weighted_mae').reset_index(drop=True)
grid_results_path = OUTPUT_DIR / 'nbeats_final_grid_results.csv'
grid_results_df.to_csv(grid_results_path, index=False)

summary_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    job_type='final_grid_summary',
    name='NBEATS_Final_Grid_Search_Summary',
    config={
        **CONFIG,
        'grid': HYPERPARAMETER_GRID,
        'total_trials': len(grid_configs),
        'baseline_reference_weighted_mae': BASELINE_REFERENCE_WMAE,
        'experiment_1_reference_weighted_mae': EXPERIMENT_1_WMAE,
        'experiment_2_reference_weighted_mae': EXPERIMENT_2_WMAE,
        'experiment_3_reference_weighted_mae': EXPERIMENT_3_WMAE,
    },
)
summary_run.log({
    'final_grid/results': wandb.Table(dataframe=grid_results_df),
    'final_grid/best_validation_weighted_mae': best_trial['best_validation_weighted_mae'],
    'final_grid/best_trial_id': best_trial['trial_id'],
})
summary_run.summary['best_trial_id'] = best_trial['trial_id']
summary_run.summary['best_epoch'] = best_trial['best_epoch']
summary_run.summary['best_validation_weighted_mae'] = best_trial['best_validation_weighted_mae']
summary_run.summary['best_params'] = {key: best_trial[key] for key in HYPERPARAMETER_GRID.keys()}
summary_run.finish()

print('--- Final grid search top 10 ---')
print(grid_results_df.head(10))
print('Best trial:')
print(best_trial)

## 8. Best grid trial diagnostics

In [ ]:
best_val_pred_original = best_trial['val_predictions']

prediction_rows = []
for series_id, (store, dept) in enumerate(series_index):
    for horizon_idx, date in enumerate(validation_dates):
        actual = float(y_val_original[series_id, horizon_idx])
        prediction = float(best_val_pred_original[series_id, horizon_idx])
        prediction_rows.append({
            'Store': int(store),
            'Dept': int(dept),
            'Date': date,
            'Weekly_Sales': actual,
            'Prediction': prediction,
            'IsHoliday': bool(val_holiday_flags[horizon_idx]),
            'AbsoluteError': abs(actual - prediction),
        })

val_predictions_df = pd.DataFrame(prediction_rows)
weekly_errors_df = (
    val_predictions_df
    .groupby('Date', as_index=False)
    .agg(Weekly_MAE=('AbsoluteError', 'mean'), IsHoliday=('IsHoliday', 'max'))
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(val_predictions_df['Weekly_Sales'], val_predictions_df['Prediction'], s=5, alpha=0.25)
axes[0].set_title('N-BEATS final grid: actual vs predicted')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Prediction')

axes[1].plot(weekly_errors_df['Date'], weekly_errors_df['Weekly_MAE'], marker='o')
axes[1].set_title('N-BEATS final grid: validation MAE by week')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

predictions_path = OUTPUT_DIR / 'final_grid_best_validation_predictions.csv'
weekly_errors_path = OUTPUT_DIR / 'final_grid_best_weekly_validation_errors.csv'

val_predictions_df.to_csv(predictions_path, index=False)
weekly_errors_df.to_csv(weekly_errors_path, index=False)

analysis_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    job_type='final_grid_evaluation',
    name='NBEATS_Final_Grid_Best_Trial_Evaluation',
    config={
        **CONFIG,
        'best_trial_id': best_trial['trial_id'],
        'best_epoch': best_trial['best_epoch'],
        'best_validation_weighted_mae': best_trial['best_validation_weighted_mae'],
        'best_params': {key: best_trial[key] for key in HYPERPARAMETER_GRID.keys()},
    },
)

evaluation_artifact = wandb.Artifact(
    name='nbeats-final-grid-evaluation',
    type='evaluation',
    description='Final N-BEATS grid-search results and best validation diagnostics. No model weights included.',
    metadata={
        'best_trial_id': int(best_trial['trial_id']),
        'best_epoch': int(best_trial['best_epoch']),
        'best_validation_weighted_mae': float(best_trial['best_validation_weighted_mae']),
    },
)
evaluation_artifact.add_file(str(grid_results_path), name='nbeats_final_grid_results.csv')
evaluation_artifact.add_file(str(predictions_path), name='final_grid_best_validation_predictions.csv')
evaluation_artifact.add_file(str(weekly_errors_path), name='final_grid_best_weekly_validation_errors.csv')

analysis_run.log({
    'final_grid/diagnostic_plots': wandb.Image(fig),
    'final_grid/predictions_sample': wandb.Table(dataframe=val_predictions_df.head(1000)),
    'final_grid/weekly_errors': wandb.Table(dataframe=weekly_errors_df),
    'final_grid/best_validation_weighted_mae': best_trial['best_validation_weighted_mae'],
})
analysis_run.log_artifact(evaluation_artifact)
analysis_run.finish()

print('Logged final grid-search results and best-trial diagnostics to W&B.')
print('No model artifact or Model Registry entry was created.')

## 9. After running this notebook

Send the top grid-search rows, best trial id, best epoch, best validation WMAE, and best hyperparameters. Then `n-beats.md` should be updated with whether final grid search improved the baseline.